<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EA%B0%95%EC%9D%98%EB%82%B4%EC%9A%A9%5D_3_LangChain%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%83%9D%EC%84%B1%EA%B3%BC_%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] LangChain을 이용한 데이터 생성과 처리


LCEL의 기본 문법인 Prompt | llm | Parser 구조에 대해 배웠습니다.   
이번 실습에서는 출력을 구조화하고, LLM을 연결하는 방법에 대해 알아봅니다.


### 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [ ]:
%pip install langchain langchain_openai dotenv rich -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 8.2 MB/s eta 0:00:00


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


### LLM 모델 불러오기

In [ ]:
from langchain.chat_models import init_chat_model

gpt_llm = init_chat_model(
    "gpt-5.2", reasoning_effort='low')

## JsonOutputParser 로 Json 형식의 출력 만들기

LLM의 출력을 구조화하면, 데이터 후처리를 하지 않고도 다른 코드와 연결할 수 있습니다.   
JSON 형식의 출력을 구성해 보겠습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

jsonparser = JsonOutputParser()

JSON 파서의 역할은 JSON 규격에 맞는 텍스트를 Dict 형식으로 변환하는 것으로,     
실제 형식에 대한 조건을 프롬프트로 전달해야 합니다.

In [ ]:
jsonparser.get_format_instructions()

'Return a JSON object.'

In [ ]:
recipe_template = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료와 조건을 이용한 환상적인
퓨전 다이닝을 만들고 싶습니다. 1가지 메뉴만 추천해주세요!
레시피에 대한 정보를 JSON 형식으로 출력해주세요.

[재료]: {ingredient}
[조건]: {condition}
''')
])

recipe_chain = recipe_template | gpt_llm | jsonparser

In [ ]:
response = recipe_chain.invoke({'ingredient':'콜라, 고수, 새우', 'condition':'디저트'})
response

{'menu_name': '콜라 카라멜 새우 브리틀 & 고수-라임 그라니타',
 'type': 'dessert',
 'fusion_concept': '남미풍 고수-라임 향을 차갑게 살린 그라니타(빙수 디저트) 위에, 콜라를 졸여 만든 카라멜로 코팅한 새우 브리틀(프랄린/캔디)을 올려 달콤-짭짤-허브 향의 대비를 만드는 퓨전 플레이트 디저트',
 'servings': 2,
 'estimated_time_minutes': 45,
 'difficulty': '중',
 'ingredients': {'cola_caramel_shrimp_brittle': [{'item': '새우(껍질 제거, 내장 제거)',
    'amount': '10마리(약 120g)'},
   {'item': '콜라', 'amount': '250ml'},
   {'item': '설탕', 'amount': '50g'},
   {'item': '버터', 'amount': '10g'},
   {'item': '소금', 'amount': '한 꼬집'},
   {'item': '식용유(굽기용)', 'amount': '약간'}],
  'cilantro_lime_granita': [{'item': '콜라', 'amount': '200ml'},
   {'item': '라임즙(또는 레몬즙)', 'amount': '20ml'},
   {'item': '고수(잎 위주)', 'amount': '10g'},
   {'item': '설탕(기호)', 'amount': '10~20g'},
   {'item': '소금', 'amount': '아주 소량(선택)'}],
  'optional_garnish': [{'item': '라임 제스트', 'amount': '약간'},
   {'item': '고수 잎', 'amount': '몇 장'}]},
 'equipment': ['냄비(소스팬)',
  '팬',
  '종이호일 또는 실리콘 매트',
  '냉동 가능한 얕은 트레이',
  '포크(그라니타 긁기용)'],
 'steps': [{'pa

In [ ]:
# Dict 구조: 추출 가능
response['menu_name']

'콜라 카라멜 새우 브리틀 & 고수-라임 그라니타'

Json으로 파싱하는 방법은 활용도가 높지만, 실행할 때마다 결과뿐만 아니라 형식도 달라진다는 문제가 있습니다.

In [ ]:
response = recipe_chain.invoke({'ingredient':'문어, 피넛버터', 'condition':'메인 요리'})
response

## Pydantic을 이용해 출력 형식 지정하기

pydantic은 데이터 형식에 제약조건을 두고 이를 준수하는지 검증하는 라이브러리입니다.


In [ ]:
from pydantic import BaseModel, Field
# pydantic 연동

class Recipe(BaseModel):
    name: str = Field(description="음식 이름")
    # name: 문자열, 설명은 "음식 이름"
    difficulty: str = Field(description="만들기의 난이도")

    origin: str = Field(description="원산지")
    ingredients: list[str] = Field(description="재료")
    # ingredients: 문자열 리스트, 설명은 "재료"

    instructions: list[str] = Field(description="조리법")
    tip: str = Field(description='실패하는 5가지 시나리오')


In [ ]:
parser = JsonOutputParser(pydantic_object=Recipe)

In [ ]:
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


해당 내용을 프롬프트에 포함합니다.

In [ ]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
재료: {ingredient}
출력 형식 조건: {instruction}''')
])

recipe_chain2 = recipe_template2 | gpt_llm | parser


In [30]:
recipe_chain2.invoke({'ingredient':'생강', 'instruction':parser.get_format_instructions()})

{'name': '생강 된장-다크초콜릿 글레이즈 가지 스테이크',
 'difficulty': '중',
 'origin': '퓨전(일본식 된장+서양식 글레이즈+동아시아 생강)',
 'ingredients': ['생강 30g(껍질 벗겨 곱게 다짐)',
  '가지 2개(두툼하게 세로로 반 갈라 칼집)',
  '된장 2큰술',
  '다크초콜릿 20g(70% 내외)',
  '간장 1큰술',
  '식초 또는 라임즙 1큰술',
  '꿀 또는 메이플시럽 1큰술',
  '올리브오일 2큰술',
  '물 3~4큰술',
  '후추 약간',
  '선택: 볶은 참깨 1작은술',
  '선택: 쪽파 또는 고수 약간',
  '선택: 바삭한 토핑(튀긴 생강채 또는 크루통) 약간'],
 'instructions': ['가지는 단면에 깊지 않게 격자 칼집을 넣고 소금을 아주 약간만 뿌려 10분 두었다가 물기를 닦는다.',
  '팬을 중불로 달구고 올리브오일을 두른 뒤 가지를 단면부터 노릇하게 4~5분 굽고 뒤집어 3~4분 더 굽는다. 팬에서 잠시 빼둔다.',
  '같은 팬에 다진 생강을 넣고 약불에서 30초~1분 향이 올라올 때까지만 볶는다(타지 않게).',
  '불을 끄고 된장, 간장, 꿀(또는 메이플), 식초(또는 라임즙), 물을 넣어 잘 풀어준다.',
  '약불로 다시 켜고 다크초콜릿을 넣어 천천히 녹이며 1~2분 걸쭉해질 때까지 졸인다. 후추로 간을 마무리한다.',
  '구워둔 가지를 팬에 다시 넣고 소스를 끼얹으며 1분 정도 글레이즈를 입힌다.',
  '접시에 담고 남은 소스를 위에 더 뿌린다. 원하면 참깨, 쪽파(또는 고수), 바삭한 토핑을 올려 질감 대비를 만든다.'],
 'tip': '실패하는 5가지 시나리오: 1) 생강을 센 불에 오래 볶아 탄맛이 나면 소스 전체가 쓴맛으로 망가진다. 2) 된장을 먼저 강불에 볶아버리면 짠맛과 비린 향이 강조되어 균형이 깨진다(반드시 물로 풀어 약불로). 3) 초콜릿을 너무 많이 넣거나 과하게 졸이면 단맛·점도가 과해져 ‘초코소스’처럼 된다(소량

partial을 통해 먼저 일부를 입력할 수도 있습니다.

In [31]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
]).partial(instruction=parser.get_format_instructions())

recipe_chain2 = recipe_template2 | gpt_llm | parser

recipe_chain2.invoke('감자')
# partial은 기본값: instruction을 다시 덮어쓸 수도 있음

{'name': '감자 미소-고추장 카라멜 브륄레(바삭 감자칩 토핑)',
 'difficulty': '중상',
 'origin': '한국·일본·프랑스 퓨전',
 'ingredients': ['감자 450g(전분 많은 품종 권장)',
  '우유 300ml',
  '생크림 200ml',
  '무염버터 30g',
  '설탕 30g(퓨레용)',
  '소금 3g',
  '흰된장(시로미소) 20g',
  '고추장 10g',
  '바닐라 익스트랙 1작은술(선택)',
  '계란 노른자 4개',
  '브륄레용 설탕 4큰술(황설탕 또는 흰설탕)',
  '감자칩 토핑용 감자 1개(얇게 슬라이스)',
  '식용유(튀김용) 적당량',
  '라임 또는 레몬 제스트 약간(선택)',
  '고춧가루 아주 소량(선택)'],
 'instructions': ['감자 450g을 껍질 벗겨 2cm 두께로 썰고 찬물에 10분 담가 전분을 일부 빼 준 뒤 물을 버린다.',
  '감자를 냄비에 넣고 우유 300ml와 물(감자가 잠길 정도)을 부어 약불에서 부드럽게 익을 때까지 끓인다(끓기 시작하면 약불 유지).',
  '감자를 건져 물기 날린 뒤 따뜻할 때 으깨고, 냄비에 다시 넣어 약불에서 1~2분 수분을 날려 퓨레를 되직하게 만든다.',
  '버터 30g, 설탕 30g, 소금 3g, 흰된장 20g, 고추장 10g(및 바닐라 선택)을 넣고 잘 섞는다.',
  '생크림 200ml를 데워(끓이기 직전) 감자 퓨레에 조금씩 넣어가며 부드럽게 풀어준다. 입자가 거칠면 체에 한 번 내려 매끈하게 만든다.',
  '볼에 노른자 4개를 풀고, 감자-크림 혼합물을 소량씩 부어가며 템퍼링한 뒤 다시 냄비로 합친다.',
  '약불에서 주걱으로 저으면서 82~84°C 정도(걸쭉해져 주걱 뒷면에 코팅되는 농도)까지 천천히 가열한다. 끓이면 안 된다.',
  '오븐을 150°C로 예열한다. 램킨에 혼합물을 나눠 담고, 깊은 팬에 램킨을 올린 뒤 뜨거운 물을 램킨 높이의 절반까지 부어 중탕 굽기한다.',
  '약 25~35분 

# LangChain Structured Output
파서를 사용하지 않고, 구조화된 출력을 생성합니다.  

In [32]:
from rich import print as rprint
structured_llm = gpt_llm.with_structured_output(Recipe)

rprint(structured_llm)

RunnableSequence(
    first=_ChatModelBinding(
        bound=ChatOpenAI(
            metadata={
                'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.13', 'langchain-openai': '1.4.1'}
            },
            output_version=None,
            profile={
                'name': 'GPT-5.2',
                'release_date': '2025-12-11',
                'last_updated': '2025-12-11',
                'open_weights': False,
                'max_input_tokens': 272000,
                'max_output_tokens': 128000,
                'text_inputs': True,
                'image_inputs': True,
                'audio_inputs': False,
                'video_inputs': False,
                'text_outputs': True,
                'image_outputs': False,
                'audio_outputs': False,
                'video_outputs': False,
                'reasoning_output': True,
                'tool_calling': True,
                'structured_output': True,
                'attachment': True,
                'temperature': False,
                'image_url_inputs': True,
                'pdf_inputs': True,
                'pdf_tool_message': True,
                'image_tool_message': True,
                'tool_choice': True,
                'tool_call_streaming': True,
                'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']
            },
            client=<openai.resources.chat.completions.completions.Completions object at 0x7ef8cbc0d340>,
            async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7ef8cbefb3e0>,
            root_client=<openai.OpenAI object at 0x7ef8f62cbf50>,
            root_async_client=<openai.AsyncOpenAI object at 0x7ef8cbcab8f0>,
            model_name='gpt-5.2',
            model_kwargs={},
            openai_api_key=SecretStr('**********'),
            openai_proxy=None,
            stream_usage=True,
            reasoning_effort='low',
            stream_chunk_timeout=120.0
        ),
        kwargs={
            'response_format': <class '__main__.Recipe'>,
            'ls_structured_output_format': {
                'kwargs': {'method': 'json_schema', 'strict': None},
                'schema': {
                    'type': 'function',
                    'function': {
                        'name': 'Recipe',
                        'description': '',
                        'parameters': {
                            'properties': {
                                'name': {'description': '음식 이름', 'type': 'string'},
                                'difficulty': {'description': '만들기의 난이도', 'type': 'string'},
                                'origin': {'description': '원산지', 'type': 'string'},
                                'ingredients': {
                                    'description': '재료',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'instructions': {
                                    'description': '조리법',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'tip': {'description': '실패하는 5가지 시나리오', 'type': 'string'}
                            },
                            'required': ['name', 'difficulty', 'origin', 'ingredients', 'instructions', 'tip'],
                            'type': 'object'
                        }
                    }
                }
            }
        },
        config={},
        config_factories=[]
    ),
    middle=[],
    last=RunnableBinding(
        bound=RunnableLambda(...),
        kwargs={},
        config={},
        config_factories=[],
        custom_output_type=<class '__main__.Recipe'>
    )
)

In [33]:
recipe_template3 = ChatPromptTemplate([
    ('system','당신은 한국 전통의 재료가 가진 다양한 맛과 특성을 활용합니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!''')
])

recipe_chain3 = recipe_template3 | structured_llm
response = recipe_chain3.invoke("생강")
rprint(response)

Recipe(
    name='생강-된장 카라멜 소스를 곁들인 구운 고구마 & 김-참깨 크럼블',
    difficulty='중',
    origin='한국 전통 재료(생강·된장·조청·김) 기반의 퓨전 디저트/안주',
    ingredients=[
        '고구마 2개(중간 크기)',
        '생강 30g(강판에 갈거나 잘게 다짐)',
        '된장 1큰술(구수한 재래된장 또는 집된장 추천)',
        '조청 또는 물엿 3큰술(없으면 꿀 2큰술+설탕 1큰술 대체)',
        '간장 1작은술',
        '버터 20g(또는 들기름 1큰술+버터 10g)',
        '우유 또는 두유 2~4큰술(농도 조절)',
        '레몬즙 또는 유자청 1작은술(산미 포인트)',
        '김 1~2장(구운 김)',
        '참깨 1큰술(볶은 것)',
        '굵은소금 한 꼬집',
        '선택: 고춧가루 아주 약간(끝 맛을 세우고 싶을 때), 후추 약간'
    ],
    instructions=[
        '고구마를 200°C 오븐에서 45~60분 굽거나(가장 달아짐), 에어프라이어 180°C 35~45분으로 익혀둡니다. (젓가락이 
쑥 들어가면 OK)',
        '팬에 버터를 약불로 녹인 뒤 생강을 넣고 1~2분만 향을 내듯 볶습니다. (갈변 전 단계에서 멈추기)',
        '된장을 넣어 30초 정도만 풀어주고, 조청(또는 물엿)을 넣어 중약불에서 2~3분 끓여 카라멜처럼 점도가 나게 
합니다.',
        '간장 1작은술을 넣고, 우유/두유를 2큰술부터 조금씩 넣어 소스 농도를 맞춥니다. (숟가락에서 천천히 흘러내리는
정도)',
        '불을 끄고 레몬즙(또는 유자청)을 넣어 산미를 살립니다. 맛을 보고 짠맛/단맛/생강의 톡 쏘는 맛을 조정합니다. 
(너무 짜면 우유, 너무 달면 레몬, 생강이 약하면 생강 추가)',
        '크럼블: 김을 잘게 부숴 팬에 참깨와 함께 30초만 볶고 굵은소금 한 꼬집을 섞어 둡니다.',
        '구운 고구마를 반 갈라 접시에 올리고, 생강-된장 카라멜 소스를 듬뿍 끼얹은 다음 김-참깨 크럼블을 뿌립니다.',
        '선택: 고춧가루를 ‘먼지처럼’ 아주 소량만 뿌리면 단짠 뒤에 매콤한 여운이 생겨 더 실험적입니다.'
    ],
    tip='실패하는 5가지 시나리오\n1) 된장이 너무 튄다/짠맛만 난다: 된장 양이 많거나 졸임이 과해 염도가 올라간 경우.
우유/두유로 풀고 조청을 1/2큰술 추가해 밸런스를 맞추세요.\n2) 생강이 맵고 씁쓸하다: 센 불에서 오래 볶아 생강이 
타거나, 생강을 과다 사용한 경우. 약불에서 짧게 향만 내고, 완성 후 레몬/유자 산미를 추가하면 ‘씁쓸함’이 
정리됩니다.\n3) 소스가 분리된다(기름 둥둥): 급격히 끓이거나 우유를 한 번에 부은 경우. 불을 끄고 천천히 저으며 
우유를 나눠 넣고, 필요하면 약불에서 30초만 재가열해 유화시키세요.\n4) 단맛이 밋밋하다: 고구마가 덜 익었거나(당화 
부족), 조청이 부족한 경우. 고구마는 꼭 충분히 ‘느리게’ 굽고, 소스에 조청을 조금 더하거나 소금 한 꼬집으로 단맛을 
세우세요.\n5) 김 크럼블이 눅눅하다: 고구마의 수분/열에 김이 금방 눅눅해짐. 크럼블은 먹기 직전에 뿌리고, 김을 팬에서
바싹 말리듯 20~30초 더 볶으면 바삭함이 유지됩니다.'
)

해당 출력은 Pydantic 클래스 형식으로 생성됩니다.   
with_structured_output 기능을 지원하지 않는 경우, PydanticOutputParser를 사용해야 합니다.

In [34]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object = Recipe)

recipe_template4 =ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
])

structured_llm2 = recipe_template4.partial(instruction = pydantic_parser.get_format_instructions()) | gpt_llm | pydantic_parser

response = structured_llm2.invoke('커피')
response

Recipe(name='커피-된장 카라멜 글레이즈를 바른 돼지목살 스테이크(미소 에스프레소 포크)', difficulty='중상 (불 조절과 농도 조절이 핵심)', origin='일본(된장/미소) + 이탈리아(에스프레소) + 미국식 글레이즈/스테이크 퓨전', ingredients=['돼지목살(또는 삼겹살) 2장(각 200~250g)', '소금 1/2작은술', '후추 약간', '식용유 1큰술', '버터 1큰술(선택)', '다진 마늘 1작은술', '된장(또는 미소) 1.5큰술', '진한 에스프레소 40ml(또는 진하게 내린 커피 60ml)', '간장 1큰술', '맛술(또는 미림) 1큰술', '꿀 또는 흑설탕 1~1.5큰술', '식초(또는 레몬즙) 1작은술', '물 1~2큰술(농도 조절용)', '파(송송) 또는 실파 약간(마무리, 선택)', '깨 약간(마무리, 선택)'], instructions=['고기 표면의 물기를 닦고 소금·후추로 밑간한 뒤 10분 정도 두어 간이 스며들게 한다.', '작은 볼에 된장, 에스프레소, 간장, 맛술, 꿀(또는 흑설탕), 식초를 섞어 글레이즈 소스를 만든다. (너무 되면 물 1큰술을 추가)', '팬을 중강불로 달군 뒤 식용유를 두르고 고기를 올려 한 면 2~3분씩 충분히 갈색이 나게 굽는다. (두께가 두꺼우면 중불로 낮춰 속까지 익힌다)', '고기를 팬 한쪽으로 밀고 빈 공간에 다진 마늘을 10~15초 볶아 향을 낸다. 버터를 넣는다면 이때 함께 넣는다.', '불을 중약불로 낮추고 만들어둔 글레이즈 소스를 팬에 붓는다. 소스가 끓기 시작하면 고기에 소스를 끼얹으며 30~60초간 빠르게 졸여 코팅한다.', '소스가 시럽처럼 윤기 있게 걸쭉해지면 불을 끄고 고기를 2~3분 휴지시킨다.', '접시에 고기를 썰어 담고 남은 소스를 위에 뿌린 뒤 파나 깨로 마무리한다. (밥, 구운 야채, 매시드 포테이토와 잘 어울림)'], tip='실패하는 5가지 시나리오: (1) 소스가 탄맛만 강함: 불이 너무 강하거나 졸이는 시간이 길면 커피 성분이 쉽게

## [실습] LLM으로 보고서 개요 생성 후 섹션별 글 작성하기

Structured Output 구조를 활용해, LLM이 주제에 대한 구획을 먼저 구성하고    
해당 구획을 반복문이나 Batch로 각각 입력하여 긴 글을 쓰도록 만들어 보세요.

1. with_structured_output을 통해 주제에 대한 구획 작성하는 체인 `outliner` 만들기
2. 섹션별 글 작성 체인 `writer` 만들기
3. 반복문이나 batch()를 통해 `outliner`의 결과물을 `writer`에 전달하기
4. 최종 결과물 합치기

In [35]:
class Sections(BaseModel):
    topic: str = Field(description="글쓰기 주제")
    sections: list[str] = Field(description="주제에 대한 세부 섹션 개요 리스트 (최대 5개 섹션)")

In [36]:
outliner = gpt_llm.with_structured_output(Sections)
outline = outliner.invoke("""
멀티모달 LLM의 발전 과정에 대한 보고서 개요와 목차를 써줘.
각각의 개요는 병렬적 작성이 가능하도록 독립적인 내용을 담아야 하고.
개요만 보고도 내용이 구체적으로 드러나야 해.
목차는 리스트로 작성해.
""")
outline

Sections(topic='멀티모달 LLM의 발전 과정(진화 로드맵) 보고서', sections=['문제 정의와 평가 프레임: 멀티모달 LLM을 ‘무엇’으로 보고 ‘어떻게’ 성능을 판단하는가(입·출력 모달리티 범위, 과제 유형, 벤치마크·지표, 안전·편향 평가 포함)', '발전 1단계—연결(Alignment) 중심: 비전/오디오 인코더+텍스트 LLM 결합 구조의 등장과 학습 레시피(프로젝터/어댑터, 동결-부분학습, 인스트럭션 튜닝, 대표 모델 계열별 특징)', '발전 2단계—엔드투엔드·통합 토크나이제이션: 모달리티를 토큰 공간에서 통합하는 접근(이미지/비디오 토큰화, 크로스어텐션 vs 단일 트랜스포머, 계산·메모리 트레이드오프)', '발전 3단계—비디오·오디오·행동으로 확장: 시간축 이해, 멀티턴 지각-추론, 툴사용/에이전트화(비디오 QA, 시청각 대화, 로봇/웹 조작 등)와 학습 데이터·피드백(RLHF/RLAIF)의 역할', '데이터·시스템·신뢰성의 병목과 향후 과제: 데이터 품질/저작권, 환각·근거성, 프롬프트/컨텍스트 한계, 경량화·온디바이스, 표준화된 벤치마크 및 규제/윤리 이슈'])

In [37]:
outline.sections

['문제 정의와 평가 프레임: 멀티모달 LLM을 ‘무엇’으로 보고 ‘어떻게’ 성능을 판단하는가(입·출력 모달리티 범위, 과제 유형, 벤치마크·지표, 안전·편향 평가 포함)',
 '발전 1단계—연결(Alignment) 중심: 비전/오디오 인코더+텍스트 LLM 결합 구조의 등장과 학습 레시피(프로젝터/어댑터, 동결-부분학습, 인스트럭션 튜닝, 대표 모델 계열별 특징)',
 '발전 2단계—엔드투엔드·통합 토크나이제이션: 모달리티를 토큰 공간에서 통합하는 접근(이미지/비디오 토큰화, 크로스어텐션 vs 단일 트랜스포머, 계산·메모리 트레이드오프)',
 '발전 3단계—비디오·오디오·행동으로 확장: 시간축 이해, 멀티턴 지각-추론, 툴사용/에이전트화(비디오 QA, 시청각 대화, 로봇/웹 조작 등)와 학습 데이터·피드백(RLHF/RLAIF)의 역할',
 '데이터·시스템·신뢰성의 병목과 향후 과제: 데이터 품질/저작권, 환각·근거성, 프롬프트/컨텍스트 한계, 경량화·온디바이스, 표준화된 벤치마크 및 규제/윤리 이슈']

In [38]:
writer_prompt = ChatPromptTemplate([
    ('human','''보고서 주제에 대해, 하나의 섹션에 대한 전문적인 글을 작성하세요.

제목은 ##, 소목차는 ###으로 쓰고, 이외의 목차 형식은 넣지 마세요.
챕터명에 숫자를 넣지 마세요.
내용은 '입니다' 와 같은 말투로 작성하세요.

---
보고서 전체 주제: {topic}
세부 섹션 주제: {section}
''')
])
writer = writer_prompt | gpt_llm | StrOutputParser()
writer.invoke({'topic':outline.topic, 'section':outline.sections[0]})

'## 문제 정의와 평가 프레임: 멀티모달 LLM을 ‘무엇’으로 보고 ‘어떻게’ 성능을 판단하는가\n\n멀티모달 LLM의 발전 과정을 체계적으로 서술하기 위해서는 먼저 “멀티모달 LLM을 무엇으로 정의할 것인가”와 “성능을 어떤 축으로 판단할 것인가”를 명확히 해야 합니다. 이는 모델 비교의 공정성을 확보하고, 로드맵에서 각 세대의 기술적 진전을 일관된 기준으로 해석하기 위한 전제입니다. 본 섹션에서는 입·출력 모달리티 범위, 과제 유형, 벤치마크 및 지표, 안전·편향 평가를 포함하는 평가 프레임을 정리하는 내용입니다.\n\n### 멀티모달 LLM의 문제 정의: 모델을 바라보는 관점 정립\n\n멀티모달 LLM은 “언어를 중심으로 다양한 모달리티 입력을 해석하고, 언어 또는 다른 모달리티로 과제 목표에 맞게 출력을 생성하는 범용 모델”로 정의하는 것이 일반적입니다. 여기에는 두 가지 관점이 공존합니다.\n\n첫째, **언어 중심 결합 모델** 관점입니다. 이미지·음성·영상 등 비언어 입력을 인코더로 임베딩화한 뒤 LLM이 이를 텍스트 토큰과 동등한 조건으로 활용하여 추론·생성하는 형태입니다. 이때 핵심은 “LLM이 멀티모달 정보를 추론에 실제로 활용하는가”이며, 단순 캡셔닝 수준을 넘어 시각적 근거 기반 추론, 다중 턴 상호작용, 도구 사용까지 포괄하는지로 범위를 규정하는 접근입니다.\n\n둘째, **통합 생성 모델** 관점입니다. 텍스트뿐 아니라 이미지·오디오 등도 직접 생성하거나, 생성 파이프라인을 호출·제어하는 형태까지 포함하는 관점입니다. 이 경우 멀티모달 LLM은 “입력 이해+출력 생성”의 양방향 능력을 가진 시스템으로 간주되며, 평가도 이해(understanding)와 생성(generation)을 분리해 설계하는 것이 적절합니다.\n\n정의의 핵심 쟁점은 “모달리티의 범위”와 “결합의 깊이”입니다. 단일 이미지-텍스트 쌍을 다루는 수준과, 영상·음성·센서·문서 레이아웃 등 복합 입력을 장시간 문맥으로 통합하는 수준은 동일한 ‘멀티모달’로 묶기 어렵기 

In [39]:
writer = writer_prompt.partial(topic=outline.topic) | gpt_llm | StrOutputParser()
# topic을 미리 채워 매개변수 1개
result = writer.batch(outline.sections)
result

['## 문제 정의와 평가 프레임: 멀티모달 LLM을 ‘무엇’으로 보고 ‘어떻게’ 성능을 판단하는가\n\n멀티모달 LLM의 발전 과정을 체계적으로 정리하기 위해서는, 먼저 멀티모달 LLM을 어떤 범주의 시스템으로 정의할지와 그 성능을 어떤 기준으로 평가할지를 명확히 해야 합니다. 동일한 모델이라도 입력·출력 모달리티의 범위, 수행 과제의 유형, 평가 벤치마크와 지표의 선택, 그리고 안전·편향 평가의 포함 여부에 따라 “성능이 좋다”는 결론이 크게 달라지기 때문입니다. 따라서 본 섹션은 멀티모달 LLM의 문제 정의를 구성요소별로 정리하고, 재현 가능하고 비교 가능한 평가 프레임을 제시하는 데 목적이 있습니다.\n\n### 멀티모달 LLM의 문제 정의: ‘언어 모델’과 ‘멀티모달 시스템’의 결합 관점\n\n멀티모달 LLM은 텍스트 중심의 언어 모델이 다양한 모달리티를 입력으로 받아 의미를 정렬하고, 텍스트 또는 다른 모달리티로 응답을 생성하는 범용 추론·생성 시스템으로 정의할 수 있습니다. 이때 핵심은 단순히 이미지 캡션처럼 특정 태스크를 수행하는 모델이 아니라, 자연어 지시를 기반으로 여러 모달 정보를 조합하여 일반화된 문제 해결을 시도하는 “지시 따르기 기반의 멀티모달 추론기”라는 점입니다. 평가 프레임은 이러한 정의를 반영하여, 모달리티 변환 능력, 조합적 추론 능력, 실제 사용 환경에서의 신뢰성과 안전성까지 포함하도록 설계되어야 합니다.\n\n### 입·출력 모달리티 범위: 무엇을 이해하고 무엇을 생성하는가\n\n평가를 위해서는 모델이 다루는 모달리티의 범위를 명시해야 합니다. 입력 측면에서는 텍스트, 이미지, 비디오, 오디오, 문서(레이아웃 포함), 3D/포인트클라우드, 센서·시계열, 코드·수식 등으로 구분할 수 있으며, 각 모달리티는 난이도와 오류 양상이 상이합니다. 예를 들어 이미지는 정적 시각 정보 이해가 핵심인 반면, 비디오는 시간 축의 사건 이해와 장면 전환, 오디오는 화자 분리와 잡음 강건성, 문서는 OCR 품질과 레이아웃 추론이 성능을 좌우합

In [40]:
draft = '\n\n'.join(result)
with open('result.md', 'w', encoding='utf-8') as f:
    f.write(draft)

print(draft[0:100])

## 문제 정의와 평가 프레임: 멀티모달 LLM을 ‘무엇’으로 보고 ‘어떻게’ 성능을 판단하는가

멀티모달 LLM의 발전 과정을 체계적으로 정리하기 위해서는, 먼저 멀티모달 LLM


<br><br>
## Runnables

LangChain 체인의 기본 구조는 `RunnableSequence` 클래스로 구성됩니다.   

이 때, 시퀀스를 구성한 llm, prompt, chain 각 모듈은 Runnables에 해당합니다.   
Runnables은 자유롭게 체인에 포함되어 결과를 연결할 수 있습니다.



이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnablePassthrough와 RunnableParallel을 이용해 체인을 구성해 보겠습니다.


<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [41]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate(["{director}의 대표 작품은 무엇입니까? 하나의 작품만 선택하고, 해당 작품에 대해 20자 이내로 설명하세요."])
chain1 = (
    prompt1
    | gpt_llm
    | StrOutputParser()
    | {'answer': RunnablePassthrough()})

response = chain1.invoke("봉준호")
response

{'answer': '**기생충**: 빈부격차를 그린 블랙코미디'}

<br><br>
### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행하여 dict 구조로 전달합니다.

In [42]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate(["색깔을 하나 알려주세요, 색깔만 출력하세요."])
prompt2 = ChatPromptTemplate(["음식을 하나 알려주세요, 음식만 출력하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(color = chain1, food = chain2)
# 개별 체인을 병렬 실행한 뒤, dict로 반환

chain3.invoke({})

{'color': '파란색', 'food': '비빔밥'}

## Assign()

RunnableParallel을 사용하면 중간 체인의 결과를 전달하여, 다음 체인의 결과를 함께 얻을 수 있습니다.   

In [43]:
prompt1 = ChatPromptTemplate(["잭슨빌은 어느 나라의 도시입니까? 나라 이름만 출력"])
prompt2 = ChatPromptTemplate(
    ["{country}의 대표적인 인물 3명을 나열하세요. 인물의 이름만 출력하세요."]
)

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(country = chain1).assign(people = chain2)
#        {    'country': '미국'           }  +    {'people': chain2(미국)}

chain3.invoke({})

{'country': '잭슨빌(Jacksonville)은 **미국(USA)**의 도시입니다. 보통 **플로리다주(Jacksonville, Florida)**를 가리키며, 플로리다주에서 인구가 가장 많은 도시로 알려져 있습니다.',
 'people': 'James Weldon Johnson  \nBob Hayes  \nFred Durst'}

<br><br><br><br><br><br><br><br>
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

In [44]:
prompt1 = ChatPromptTemplate([
    "{city}는 어느 나라의 도시인가요? 나라 이름만 출력하세요."])
prompt2 = ChatPromptTemplate([
    "{country}의 유명한 인물은 누가 있나요? {num} 명의 이름을 나열하세요. 사람 이름만 ,로 구분하여 나열하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()

chain2 = (
    RunnablePassthrough.assign(country = chain1)
    # 입력받은 city, num에 country를 추가하여 전달
    # city, num          +    country

    | prompt2
    # country, num을 받아 실행
    | gpt_llm
    | StrOutputParser()
)

print(chain2.invoke({"city": "잭슨빌", "num": "3"}))

George Washington, Abraham Lincoln, Martin Luther King Jr.


<br><br>
assign을 여러 개 연결할 수 있습니다.

In [45]:
chain4 = (prompt2
    | gpt_llm
    | StrOutputParser())

chain3 = RunnablePassthrough.assign(country = chain1).assign(people = chain4)
#        {'city', 'num'}      +

chain3.invoke({"city": "부에노스 아이레스", "num": "3"})

{'city': '부에노스 아이레스',
 'num': '3',
 'country': '아르헨티나',
 'res': '리오넬 메시, 디에고 마라도나, 호르헤 루이스 보르헤스'}

<br><br><br>JsonOutputParser를 쓴다면 아래와 같이 만들 수도 있습니다.

In [46]:
prompt1 = ChatPromptTemplate(
    ["영화 배우 한명과 대표작 하나를 출력하세요. json 형식으로 출력하고, 각 항목은 actor, movie로 표시하세요."])
prompt2 = ChatPromptTemplate(["{actor}는 {movie}에서 어떤 역할을 했습니까?"])

chain1 = prompt1 | gpt_llm | JsonOutputParser()
chain2 =(
     chain1 | prompt2 | gpt_llm | StrOutputParser()
)
chain2.invoke({})

'송강호는 영화 **《기생충》(2019)**에서 **김기택** 역을 맡았습니다.  \n김기택은 **기우(최우식)**, **기정(박소담)**, **충숙(장혜진)**의 아버지이자, 박사장(이선균)네 집에 들어가 **운전기사**로 일하게 되는 인물입니다.'

In [47]:
chain3 = prompt2 | gpt_llm | StrOutputParser()

chain4 = chain1.assign(result = chain3)

chain4.invoke({})

{'actor': '송강호',
 'movie': '기생충',
 'result': '송강호는 영화 **〈기생충〉(2019)**에서 **기택** 역을 맡았습니다. 기택은 **김가족의 아버지**로, 일자리를 잃고 어렵게 살다가 박사장(이선균)네 집에 **운전기사**로 들어가면서 이야기가 전개됩니다.'}